# Lindblad example
Welcome! This is a notebook that will guide you through the Lindblad.py package. We will have a look at a one spin-1/2 and a two spin-1/2 system to see how they behave and what functions are available. Feel free to copy this notebook and experiment with it.



### Import libraries

You will need the usual packages as well as `Qutip`, `scipy`, `copy` and `joblib`. So far the Lindblad package has to be imported separately but it will be added to the SpinFinity main branch sooner or later.

You can find more information on the Qutip package and its installation here: https://qutip.readthedocs.io/en/latest/index.html.
The code uses the mesolve function to solve the Lindblad equation.

Scipy is needed for the Fourier transformation function. Copy and joblib are needed for parallelization in the tip field sweep function.

In [1]:
# import python libraries 
import sys
import os
import subprocess
import matplotlib.pyplot as plt
import numpy as np
import scipy.constants as const
import plotly
import scipy    # NEW
from copy import deepcopy   # NEW
from joblib import Parallel, delayed    # NEW

# import spin code and Lindblad package (not in main yet)
from spinfinity import *
from spinfinity import Lindblad as lind     # NEW

%load_ext autoreload
%autoreload 2

# import qutip code
from qutip import *     # NEW

## Overview of the available functions

The following functions are available in `Lindblad.py` and will be explained below:
- collapseOperators
- calcTimeEvolution
- plotTimeEvolution
- calcDecayTipfieldsweep
- plotDecayTipfieldsweep
- plotTipZeemanDiagramm
- calcDensityMatrix
- plotDensityMatrix
- calcTunnelCurrent_time
- plotTunnelCurrent_time
- calcFourierFreqs
- plotFourierFreqs

## One spin 1/2

We begin with a spin-1/2 in an external field. Then we look at the most important functions of Lindblad.py to understand the code structure. The expected eigenstates up and down are also displayed with `vis.plotEigenMatrix`.

In [3]:
# initialize SpinSys object with magnetic fields and temperature
spinsys1 = SpinSys([0.5])
spinsys1.B = np.array([0, 0, 0.5])  # T
spinsys1.BTip = np.array([0, 0, 0]) # T
spinsys1.T = 1    # K

# define tunneling parameters
spinsys1.G_ss = [2e-9]*spinsys1.NSpins  # S
spinsys1.G_St = 0.2e-9
spinsys1.G_tt = 1
spinsys1.V_DC = 0

# plot EigenMatrix
hamil.calc_EigStates(spinsys1)
bath.calc_Lifetimes(spinsys1)
hamil.plot_EigenMatrix(spinsys1);

# QUESTION tip polarization -> only relevant for rates when there is a current

/home/paul-greule/Desktop/Git Projects/spinfinity/spinfinity/Bath.py:409: RuntimeWarning: invalid value encountered in scalar divide
  delta_E


### Calculate and plot < Sz > (t)

To solve the Lindblad equation with `lind.calcTimeEvolution`, the system needs collapse operators. These are generated by `lind.collapseOperators` and stored as attributes of the `spinsys` object.

Because the collapse operators depend on the Hamiltonian's eigenstates and the rates, it is safer to recompute them (with the default parameter recalcRates=True) after any parameter change that affects the Hamiltonian or the tunneling rates.

To get time-dependent results, always run `lind.calcTimeEvolution` first. It stores the following attributes:

- `spinsys.timeEvolution_result`: the QuTiP `mesolve` result that contains the time values and the system's density matrix at each time step.
- `spinsys.timeEvolution_spinidx`: the indices of the spins for which the time evolution was calculated. In our case it is 0 and by default all spins are considered.
- `spinsys.timeEvolution_Saxis`: the spin operator of which the expectation values were calculated based on the density matrices (default is `Sz`).

From these attributes, the time evolution, density matrices, Fourier transformation and current can be plotted. So keep in mind that the last  `lind.calcTimeEvolution` that you ran is the base for the other time-dependent functions.

In [4]:
# calc and plot time evolution
psi_exc = np.array([0, 1])    # define excited state
lind.calcTimeEvolution(spinsys1, psi_0=psi_exc, timeRange=[0, 1e-6], N_times=200)   # calculate the time evolution for a specific time range and initial state. 
lind.plotTimeEvolution(spinsys1);   # plot the calculated time evolution

AttributeError: module 'spinfinity.SpinHamiltonian' has no attribute 'calcEigEnergies'

We see an exponential relaxation of a fully excited spin state towards thermal equilibrium.

Let's try the same thing with `T=0.0001` K and observe the end state. Notice that we need a longer time frame here to capture the whole relaxation process as it is slower for lower temperatures.

In [ ]:
# same thing with very low temperature TODO check decay time depends on temperature?
spinsys1.T = 0.0001 # K
lind.collapseOperators(spinsys1)
lind.calcTimeEvolution(spinsys1, psi_0=psi_exc, timeRange=[0, 4e-6], N_times=200)
lind.plotTimeEvolution(spinsys1);

For very low temperatures, the spin relaxes towards the ground state. We can now investigate what a different initial state leads to. 

We start in a superposition of the eigenstates and return to `T = 1` K. The relaxation is still exponential but the initial value of < Sz > should be different. For multiple coupled spins, this could result in oscillating behaviour, but for this single-spin case, the main visible difference is the initial condition. The end state is the same as in the previous example with the same temperature.

In [ ]:
# higher temperature again but superposition initial state
spinsys1.T = 1
lind.collapseOperators(spinsys1)

psi_mix = np.sqrt(1/3)*np.array([1, 0]) + np.sqrt(2/3)*np.array([0, 1]) # define superposition state
lind.calcTimeEvolution(spinsys1, psi_0=psi_mix, timeRange=[0, 1e-6], N_times=200)
result1_mix = lind.plotTimeEvolution(spinsys1);     # plot and store result for fitting

### Fit to check the time evolution simulation

To be sure that the time evolution code produces credible results, we can fit the $S_z(t)$ curve to an exponential function $S(t)=A_0*\exp{-\frac{t}{\tau}}+S_T$. We get the time and magnetization through the returned dict from the plot function as usual.

$\tau$ should correspond to the relaxation time $T_1$ and $S_T$ to the steady-state equilibrium magnetization from the populations.

In [ ]:
# get time and Sz values from return dict
time_data = np.array(result1_mix['time_values'])
Sz_data = result1_mix['expectation_values'][0]  # take 0 entry for spin 0

# calculate and print T1 and ST from non-Lindblad code
hamil.calcEigEnergies(spinsys1)
bath.calcRates(spinsys1)
bath.calcPopulations(spinsys1)
bath.calcLifetimes(spinsys1)

print(f'T1_tot: {spinsys1.T1_tot}')

W01 = spinsys1.Rates_Summed[0, 1]
W10 = spinsys1.Rates_Summed[1, 0]
T1_expect = 1 / (W01 + W10)     # TODO check T1 calculation
ST_expect = spinsys1.Populations[0]-spinsys1.Populations[1]   # calculate the equilibrium magnetization from the populations

print(f'expected S_T = {ST_expect:.5f}')
print(f'expected T1 = {T1_expect*1e9:.5f} ns')


# exponential fit
def exp_func(t, A, tau, S_T):   # fitting function
    return A * np.exp(-t / tau) + S_T

# set initial parameter guesses
A0 = -0.5 
tau0 = T1_expect
S_T0 = ST_expect
p0 = [A0, tau0, S_T0]

# fit and print results
popt, pcov = scipy.optimize.curve_fit(exp_func, time_data, Sz_data, p0=p0)
A_fit, tau_fit, S_T_fit = popt
print(f"Fitted parameters: A={A_fit:.5f}, tau={tau_fit*1e9:.5f}ns, S_T={S_T_fit:.5f} (rounded)")

# Plot fit
fig, ax = plt.subplots()
ax.plot(time_data, Sz_data, label='Data')
ax.plot(time_data, exp_func(time_data, *popt), 'r--', label='Fit')
ax.set_xlabel('Time (s)')
ax.set_ylabel('<Sz>')
ax.set_ylim([-1, 1])
ax.legend()
plt.show()
# TODO tau is not T1 WHY? Plot looks like fit value is correct (half decay at tau)

print(S_T_fit-Sz_data[0])   # should be -A_fit TODO delete

### Tip field sweep

We can now plot a tip field sweep using `lind.plotDecayTipfieldsweep` to see if a different behaviour occurs for a non-zero tip field. But our single spin is unfortunately too boring for that. 

In general one needs to be careful when choosing the number of time and BTip field steps (`N_times`and `N_BTip`). The plot can be misleading and mesolve possibly doesn't converge if there are too little steps. On the other hand, increasing the steps also increases the calculation time.

In [ ]:
lind.plotDecayTipfieldsweep(spinsys1, psi_0=psi_exc, timeRange = [0, 1e-6], N_times=200, BTipRange = [-0.1, 0.1], N_BTip=200);

## Two coupled spin-1/2

At this point, a new SpinSys object will be initialized. It has a different name but to be sure that no parameters get mixed up, better restart the kernel and only run the import cell.

We will now have a look at a more dynamic system of two coupled spin-1/2 with B = [0, 0, 0.5] T external field, a tip field of BTip = [0.01, 0, 0.01] T and 0.5 GHz exchange coupling. We can first have a look at the Hamiltonian of the system and plot the eigenmatrix and tip field Zeeman diagram. We will see that with this non-zero tip field, there is a bit of detuning but the eigenstates |1> and |2> are still mixed spin states.

In [ ]:
# initialize SpinSys object 
spinsys2 = SpinSys([0.5, 0.5])
spinsys2.B = np.array([0, 0, 0.5])  # T
spinsys2.BTip = np.array([0.01, 0, 0.01]) # T
spinsys2.T = 0.3    # K
J_GHz = 0.5
spinsys2.Jvector[0, :] = np.array([1, 1, 1])*J_GHz*spinsys2.GHztomeVConversion    # 0.5 GHz exchange coupling

# define tunneling parameters
spinsys2.G_ss = [2e-9]*spinsys2.NSpins 
spinsys2.G_st = 0.2e-9
spinsys2.G_tt = 1
spinsys2.V_DC = 0   # no voltage bias

# plot EigenMatrix
hamil.calcEigStates(spinsys2)
hamil.plotEigenMatrix(spinsys2);
lind.plotTipZeemanDiagramm(spinsys2, BTipRange=[-0.1, 0.1], N_BTip=200);    # plot tip field Zeeman diagram specifying BTip range and number of tip field data points

# calculate Lindblad operators
lind.collapseOperators(spinsys2);

### Time evolution and Fourier transformation

Let's plot the time evolution of the spin system using `lind.calcTimeEvolution` and `lind.calcTimeEvolution` that we already know. For this we set the initial state to [0, 0, 0, 1] (in the spin basis). This is an eigenstate of the Hamiltonian as we see in the eigenmatrix. A density matrix (for example as a numpy 4x4 array) can also be given to the function as the initial state.

In [ ]:
psi_evec = np.array([0, 0, 0, 1])   # excited state (eigenstate)
lind.calcTimeEvolution(spinsys2, psi_evec, timeRange=[0, 2e-6], N_times=200)
lind.plotTimeEvolution(spinsys2);

For an eigenstate, one can observe that no oscillations occur.

The next step will be to look at the behaviour for an initial state that is not an eigenstate of the Hamiltonian. For this we can choose [0, 0, 1, 0] which mixes two spin states. Now we expect oscillations of < Sz > (t). We first calculate one time evolution result with few time steps and then a second result with more steps to see what difference this makes. This last result gets plotted twice, once the full time range and once only spin 0 with a zoom into the first ns. 

In [ ]:
psi_mix = np.array([0, 0, 1, 0])  # define mixed state 

# calculate and plot time evolution for few time steps for demonstration
lind.calcTimeEvolution(spinsys2, psi_mix, timeRange=[0, 2e-6], N_times=100)  
lind.plotTimeEvolution(spinsys2);

# calculate and plot time evolution for more time steps
Nt = 5.744375e8*2e-6*8    # number of time steps to have 8 time steps per oscillation period for better visualization (already knowing the oscillation frequency)
print(Nt)
lind.calcTimeEvolution(spinsys2, psi_mix, timeRange=[0, 2e-6], N_times=int(Nt))    # calculate time evolution for more time steps
lind.plotTimeEvolution(spinsys2);       # plot full result
lind.plotTimeEvolution(spinsys2, timeRange=[0, 10e-9], spin_idxToPlot=[0]);    # plot result only for spin 0 and for a specific time range

The result will also be Fourier transformed using `lind.plotFourierFreqs`. Note that for the Fourier peaks to be sharp, the time evolution result has to contain enough oscillation periods. 

We expect that the main frequency that is present corresponds to the exchange coupling of 500 MHz and some detuning $\delta$ from the tip field: $f_{calc}=\sqrt{J^2+\delta^2}$. There can also be frequency peaks from the exponential decay. In the zoom in plot just above we see around 5.7 oscillations in 10ns, which corresponds to a frequency of 570 MHz that we can expect.

In [ ]:
# calculate and plot Fourier transformation from time evolution result
mix_freqs = lind.plotFourierFreqs(spinsys2)
freq_peaks = mix_freqs['frequency_peaks']
print(f'frequencies of the oscillations: \n spin 0:{freq_peaks[0]*1e-6} MHz \n spin 1:{freq_peaks[1]*1e-6} MHz \n')

# calculate expected oscillation frequency from J and detuning through BTip
delta = spinsys2.BTip[2]*spinsys2.gfactorvector[0]*spinsys2.muB*spinsys2.meVtoGHzConversion # g*muB*BTip_z in GHz
f_calc = np.sqrt((delta)**2 + (J_GHz)**2)  # sqrt(J^2+delta^2) in GHz
print(f'f_calc = {f_calc*1e3} MHz')  # TODO looks good but small discrepancy with ft result

### Density matrices    

To visualize the full density matrix calculated via `lind.calcTimeEvolution`and not only the Sz expectation value, we can plot the density matrix at specified times. The function `lind.plotDensityMatrix` plots the density matrices at the time values of the time evolution that are closest to the requested times it gets. Let's look at the density matrices at $t=0$, after roughly 1/2 oscillation and one oscillation and towards the end of the oscillation pattern at $t=1.5 \, \mu s$.

Red bars in the plot means negative values, grey means exactly 0 and blue positive values. We will see that off-diagonals also start oscillating and that the system relaxes towards an equilibrium with more ground state contributions.

In [ ]:
times_densmats = [0, 1/2/574e6, 1/574e6, 1.5e-6]  # define the time values for which to plot the density matrices
lind.plotDensityMatrix(spinsys2, time_type="value", time_value=times_densmats);

### Tip field sweep

And some last tip field sweeps... first for the shorter time frame (10ns) and then for a longer time period. You can compare the behaviour showed in the sweep to the result above with BTip_z = 0.01 T. The sweep is performed for spin 0 by default but you can change that with the parameter `spinidx`. The calculation might take some time depending on your computer.

If you get an `IntegratorException`, try increasing `N_times` or `nsteps` through the argument options={'nsteps':1000} (1000 is default).

In [ ]:
lind.plotDecayTipfieldsweep(spinsys2, psi_0=psi_mix, timeRange = [0, 1e-8], N_times=500, BTipRange = [-0.05, 0.05], N_BTip=100);

In [ ]:
lind.plotDecayTipfieldsweep(spinsys2, psi_0=psi_mix, timeRange = [0, 2e-6], N_times=1000, BTipRange = [-0.05, 0.05], N_BTip=200, options={'nsteps':5000});

### Tunneling current

The last function to introduce is `lind.plotTunnelCurrent_time`. Its calculation works in a similar way as `bath.calcTunnelCurrent`. It calculates the current from the occupations of the different spin states (the diagonal entries of the density matrix at each time step) and the rates from and to each state. So keep in mind that this calculation also works without voltage like in this example. The results have to be interpreted with some caution when comparing it to an experiment. 

In [ ]:
lind.plotTunnelCurrent_time(spinsys2);  # plot the tunnel current
lind.plotTunnelCurrent_time(spinsys2, timeRange=[0, 10e-9]);    # plot the tunnel current for a shorter time range